# Thai Air Intelligence — XGBoost + LightGBM PM2.5 Trainer

**Run this in Google Colab only. It is NOT deployed to Vercel.**

This notebook trains a per-province XGBoost + LightGBM ensemble on the
`daily_summary` table, then **distils** each ensemble into a lightweight linear
surrogate (`coef` + `intercept`) that the Vercel function `api/ml/forecast.py`
can evaluate in <10s without importing any GBM library.

It writes one row per province into `public.model_registry`:

- `model_name`   = `xgb-lgbm-ensemble-v1`
- `mae / rmse / r2` = hold-out metrics of the ensemble
- `model_params` (jsonb) = `{ features, feature_importance, coef, intercept, diurnal, pm25_floor, horizon_days }`

> ⚠️ **Run this BEFORE hitting `/api/ml/forecast`.** If `model_registry` has no
> active row with `model_params.coef`, the endpoint returns `0 rows`.

## Setup
1. Set your `SUPABASE_URL` and `SUPABASE_SERVICE_ROLE_KEY` below
   (use Colab Secrets 🔑 or paste when prompted — never commit them).
2. Run all cells top to bottom.

In [ ]:
# 1 — Install training deps (Colab only; these are intentionally NOT in requirements.txt)
!pip -q install xgboost lightgbm scikit-learn requests numpy

In [ ]:
# 2 — Config & Supabase REST client
import os, json, math, getpass
import numpy as np
import requests

# Prefer Colab Secrets; fall back to a prompt. NEVER hard-code keys here.
try:
    from google.colab import userdata
    SUPABASE_URL = userdata.get("SUPABASE_URL")
    SERVICE_KEY  = userdata.get("SUPABASE_SERVICE_ROLE_KEY")
except Exception:
    SUPABASE_URL = SERVICE_KEY = None

SUPABASE_URL = SUPABASE_URL or input("SUPABASE_URL (https://xxxx.supabase.co): ").strip()
SERVICE_KEY  = SERVICE_KEY  or getpass.getpass("SUPABASE_SERVICE_ROLE_KEY: ").strip()
SUPABASE_URL = SUPABASE_URL.rstrip("/")

MODEL_NAME   = "xgb-lgbm-ensemble-v1"
HORIZON_DAYS = 7
PM25_FLOOR   = 1.0
# Feature order is a CONTRACT with api/ml/forecast.py — keep them identical.
FEATURES = ["lag1", "lag7", "roll7", "roll14", "doy_sin", "doy_cos",
            "hotspot", "burning", "temp", "humidity", "wind"]

def _h(write=False):
    h = {"apikey": SERVICE_KEY, "Authorization": f"Bearer {SERVICE_KEY}",
         "Content-Type": "application/json"}
    if write:
        h["Prefer"] = "resolution=merge-duplicates,return=minimal"
    return h

def rest_get(path, params):
    r = requests.get(f"{SUPABASE_URL}/rest/v1/{path}", headers=_h(), params=params, timeout=30)
    r.raise_for_status()
    return r.json()

print("Connected to", SUPABASE_URL)

In [ ]:
# 3 — Load daily_summary and build supervised (X, y) rows per province
def fetch_province(pid, limit=1500):
    rows = rest_get("daily_summary", {
        "select": "date,pm25_mean,hotspot_count,is_burning_season,temp_mean,humidity_mean,wind_speed_mean",
        "province_id": f"eq.{pid}",
        "order": "date.asc",
        "limit": str(limit),
    })
    return [r for r in rows if r.get("pm25_mean") is not None]

def num(v, d=0.0):
    try:
        return float(v) if v is not None else d
    except (TypeError, ValueError):
        return d

def make_dataset(rows):
    """Predict next-day pm25_mean from lagged/seasonal/exogenous features."""
    pm = [num(r["pm25_mean"]) for r in rows]
    X, y = [], []
    for i in range(len(rows)):
        if i < 14 or i + 1 >= len(rows):
            continue
        dt = rows[i + 1]["date"]  # target day
        doy = (np.datetime64(dt) - np.datetime64(dt[:4] + "-01-01")).astype(int) + 1
        feat = {
            "lag1": pm[i], "lag7": pm[i - 7],
            "roll7": float(np.mean(pm[i - 6:i + 1])),
            "roll14": float(np.mean(pm[i - 13:i + 1])),
            "doy_sin": math.sin(2 * math.pi * doy / 365.25),
            "doy_cos": math.cos(2 * math.pi * doy / 365.25),
            "hotspot": num(rows[i].get("hotspot_count")),
            "burning": 1.0 if rows[i].get("is_burning_season") else 0.0,
            "temp": num(rows[i].get("temp_mean"), 28.0),
            "humidity": num(rows[i].get("humidity_mean"), 60.0),
            "wind": num(rows[i].get("wind_speed_mean"), 5.0),
        }
        X.append([feat[f] for f in FEATURES])
        y.append(pm[i + 1])
    return np.array(X, dtype=float), np.array(y, dtype=float)

# Province ids: pull distinct from the table (falls back to TH-30..TH-49).
prov_rows = rest_get("daily_summary", {"select": "province_id", "limit": "5000"})
PROVINCES = sorted({r["province_id"] for r in prov_rows}) or [f"TH-{n}" for n in range(30, 50)]
print(len(PROVINCES), "provinces:", PROVINCES[:5], "...")

In [ ]:
# 4 — Train XGB + LGBM per province, distil into a linear surrogate
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# A neutral diurnal shape used to expand daily means into 168 hourly points
# downstream. Peaks overnight/early-morning, troughs midday.
DIURNAL = [round(1.0 + 0.16 * math.exp(-((h - 6) ** 2) / 12)
                     + 0.10 * math.exp(-((h - 20) ** 2) / 16) - 0.08, 3)
           for h in range(24)]

def train_one(pid):
    rows = fetch_province(pid)
    X, y = make_dataset(rows)
    if len(y) < 60:                       # not enough history to train
        return None
    split = int(len(y) * 0.8)
    Xtr, Xte, ytr, yte = X[:split], X[split:], y[:split], y[split:]

    xgb = XGBRegressor(n_estimators=300, max_depth=4, learning_rate=0.05,
                       subsample=0.9, colsample_bytree=0.9, random_state=42)
    lgbm = LGBMRegressor(n_estimators=300, max_depth=5, learning_rate=0.05,
                         subsample=0.9, colsample_bytree=0.9, random_state=42, verbose=-1)
    xgb.fit(Xtr, ytr); lgbm.fit(Xtr, ytr)

    ens = lambda M: (xgb.predict(M) + lgbm.predict(M)) / 2.0
    pred_te = ens(Xte)
    mae  = float(mean_absolute_error(yte, pred_te))
    rmse = float(math.sqrt(mean_squared_error(yte, pred_te)))
    r2   = float(r2_score(yte, pred_te)) if len(yte) > 1 else 0.0

    # Distil the (non-linear) ensemble into a linear surrogate the lightweight
    # endpoint can evaluate: fit Ridge to the ENSEMBLE'S predictions on all X.
    surrogate = Ridge(alpha=1.0).fit(X, ens(X))

    # feature_importance: average the normalised gains of both GBMs.
    fi_xgb = np.asarray(xgb.feature_importances_, dtype=float)
    fi_lgb = np.asarray(lgbm.feature_importances_, dtype=float)
    fi = (fi_xgb / (fi_xgb.sum() or 1) + fi_lgb / (fi_lgb.sum() or 1)) / 2.0
    feature_importance = {f: round(float(v), 4) for f, v in zip(FEATURES, fi)}

    model_params = {
        "schema_version": 1,
        "features": FEATURES,
        "feature_importance": feature_importance,
        "coef": [round(float(c), 6) for c in surrogate.coef_],
        "intercept": round(float(surrogate.intercept_), 6),
        "diurnal": DIURNAL,
        "pm25_floor": PM25_FLOOR,
        "horizon_days": HORIZON_DAYS,
    }
    return {"province_id": pid, "training_rows": int(len(y)),
            "mae": round(mae, 3), "rmse": round(rmse, 3), "r2": round(r2, 4),
            "model_params": model_params}

results = []
for pid in PROVINCES:
    try:
        out = train_one(pid)
        if out:
            results.append(out)
            print(f"{pid}: rows={out['training_rows']:4d}  MAE={out['mae']:.2f}  R2={out['r2']:.3f}")
        else:
            print(f"{pid}: skipped (insufficient history)")
    except Exception as e:
        print(f"{pid}: ERROR {e}")
print(f"\nTrained {len(results)} provinces.")

In [ ]:
# 5 — Upsert results into model_registry (one active row per province)
from datetime import datetime, timezone

def upsert_models(results):
    now = datetime.now(timezone.utc).isoformat().replace("+00:00", "Z")
    payload = [{
        "model_name": MODEL_NAME,
        "province_id": r["province_id"],
        "trained_at": now,
        "training_rows": r["training_rows"],
        "mae": r["mae"], "rmse": r["rmse"], "r2": r["r2"],
        "is_active": True,
        "model_params": r["model_params"],
    } for r in results]
    resp = requests.post(
        f"{SUPABASE_URL}/rest/v1/model_registry",
        headers=_h(write=True),
        params={"on_conflict": "model_name,province_id"},
        data=json.dumps(payload),
        timeout=60,
    )
    resp.raise_for_status()
    return len(payload)

if results:
    n = upsert_models(results)
    print(f"✅ Upserted {n} rows into model_registry as '{MODEL_NAME}'.")
    print("model_registry now has feature_importance — /api/ml/forecast will return rows.")
else:
    print("⚠️ No models trained — nothing written. Check that daily_summary has history.")